# C.5 花更多計算值得嗎？


生成一份80-token長解答，或四份各20-token短解答，兩者生成文字都共80個token。但四份還要選出最後答案，如果驗證每份又用5個token，總量就不同。前置是[多候選與命中](https://birdhackor.github.io/tiny-perceptron-vlm/C.3.html)和[選擇流程](https://birdhackor.github.io/tiny-perceptron-vlm/C.4.html)。本節先把預算攤開，再說如何比較品質與等待時間，避免把「候選變多」直接當成免費改善。

這裡用一個人工規劃：假設選擇器本身也是模型，每查一份候選生成5個驗證token。單份方案花80+5=85，四份方案花4×20+4×5=100。這些是規劃數字，不是測得的模型耗時；算術驗證器若不用模型生成，就不會有這種驗證token，但仍有實際執行時間。必須先說明成本單位，不能把秒數與token數不加說明地相加。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
plans = [
    {"candidates": 1, "answer_tokens_each": 80},
    {"candidates": 4, "answer_tokens_each": 20},
]
verify_tokens_each = 5
for plan in plans:
    n = plan["candidates"]
    generated = n * plan["answer_tokens_each"]
    verified = n * verify_tokens_each
    print("候選", n, "生成", generated, "驗證", verified, "總token", generated + verified)

輸入是兩個候選數與單份答案長度，另設固定驗證長度5。輸出是候選1、生成80、驗證5、總85；候選4、生成80、驗證20、總100。`n*...`將每份成本乘上候選數，沒有生成文字、計時或測準確率。相同生成預算不等於相同整體預算，這是這段算術實際支持的結論。

真正的test-time compute（測試時算力）是回答當下額外花的運算，例如更長步驟、多份候選與驗證。模型權重是訓練會調整的模型內數值，決定模型如何計算各候選的分數。本節可以把這些數值保持固定，只改回答流程，而不是再次訓練。比較時可以固定總token上限，調整候選數與每份長度，再測最終答案正確率、集合命中、選擇失敗及實際花費。若增加預算，只在很容易的題目多寫相同答案，收益可能很小。

token數也不是秒數。四份候選可一起計算，或分次計算；長答案依序逐token生成。計算下一個token的候選分數時，模型會參考輸入與已生成的內容；這種依相關程度取用已有內容的運算叫注意力。答案愈長，已有的token愈多，每一步可能需要參考的範圍也愈大。[KV快取](https://birdhackor.github.io/tiny-perceptron-vlm/16.3.html)保存先前已算出的部分結果，減少重算，仍無法讓每個token的成本永遠固定。應在同一硬體、同樣輸入條件下量從請求到完成的時間，區分總運算成本和使用者等待。這段程式不提供延遲數字，所以不能用85/100比例預測速度。

正式比較每份短答最多新增8個token，每份步驟最多48個，沒有固定相同總生成預算。下表是全部24題的實際生成總token，包含EOS（表示生成結束的專用token，也計入生成數量），與各題候選batch生成時間的總和；同題候選一起計算，不能把秒數當作逐份串行成本：

| k | 短答生成token | 步驟生成token | 短答生成秒數 | 步驟生成秒數 |
| --- | --- | --- | --- | --- |
| 1 | 51 | 512 | 0.239 | 1.160 |
| 2 | 98 | 1,034 | 0.135 | 1.169 |
| 4 | 202 | 2,072 | 0.122 | 1.186 |
| 8 | 400 | 4,132 | 0.120 | 1.190 |

步驟k=8的命中14/24比k=1的11/24多三題，卻用了約八倍生成token；多數決只從11/24升至12/24。這是增加預算的觀察，沒有證明等預算下最佳配方。短答k=8的token較多、時間卻較短，受batch與先後執行條件影響；每格只有一次流程，不能當穩定速度排名或把token比值當時間比值。

計時欄包含模型逐token生成及採樣；沒有包含事先將問題與對話標記整理成輸入ID，也沒有包含事後把生成ID還原成可讀文字、投票或程式驗證。這裡排除的是ID還原文字，不是模型逐token生成的運算。算術驗證沒有新增模型token，仍有CPU工作。

要量一個已啟動服務的回答延遲，應從收到問題開始，到使用者取得最後文字結束。這段包括輸入整理、模型生成、回答選擇或驗證，以及輸出文字還原；表格只量中間的生成，不能直接當成整次請求的等待時間。若服務尚未啟動，首次載入模型的冷啟動時間可另外記錄，避免和後續請求混在一起。

另有一種成本發生在回答之前：準備環境、訓練、保存與上傳模型。本輪整項訓練、評估與本機保存共45.71秒，未含機器與環境準備、也未含Hugging Face上傳。這是完成實驗的成本，與已啟動服務回答一道題的延遲分開計算。

練習只把verify_tokens_each改成0。先預測兩個方案總token都變80，再執行核對；然後說明這只代表驗證不再新增生成token，並不代表驗證無耗時、四份候選一定更準，或兩方案等待時間必然相同。若要做品質曲線，下一步應取得真實模型在固定測試題上的結果。
